# Evaluate Models

In [1]:
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# Load fine-tuned model and tokenizer
model_path = "../tinyllama-mcq-lora"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)
model.eval()

# Load test data
with open("../data/final_mcqs_test.jsonl") as f:
    test_data = [json.loads(line) for line in f]

correct = 0
total = 0

for item in tqdm(test_data):
    prompt = item["prompt"].strip()

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Only keep generated portion
    generated = decoded[len(prompt):].strip()

    match = re.search(r"<answer>([A-D])</answer>", generated)
    predicted_answer = match.group(1) if match else None
    gold_answer = re.search(r"<answer>([A-D])</answer>", item["completion"])
    gold_answer = gold_answer.group(1) if gold_answer else None

    if predicted_answer == gold_answer:
        correct += 1
    total += 1

accuracy = correct / total if total > 0 else 0
print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")


/opt/anaconda3/envs/AIHC/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


  0%|          | 0/123 [00:00<?, ?it/s]/opt/anaconda3/envs/AIHC/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
  0%|          | 0/123 [01:00<?, ?it/s]


KeyboardInterrupt: 